In [ ]:
from pathlib import Path
ROOT = next(p for p in [Path().resolve(), *Path().resolve().parents] if (p / "src").is_dir())

def eades_order(C):
    """
    Da capire: per ora teniamoci il fatto che rende C nilpotente
    """
    
    A = (C == 1)
    n = A.shape[0]
    
    
    
    alive = np.ones(n, bool)
    indeg  = A.sum(1).astype(float)   # A[i,:] = input di i
    outdeg = A.sum(0).astype(float)   # A[:,i] = prodotti di cui i e' input
    
    
    
    
    s_head, s_tail = [], []
    while alive.any():
        moved = True
        while moved:
            moved = False
            for v in np.where(alive & (outdeg == 0))[0]:   # pozzi -> coda
                if alive[v]:
                    s_tail.append(v); alive[v] = False
                    indeg[A[:, v]] -= 1; outdeg[A[v, :]] -= 1; moved = True
            for v in np.where(alive & (indeg == 0))[0]:    # sorgenti -> testa
                if alive[v]:
                    s_head.append(v); alive[v] = False
                    indeg[A[:, v]] -= 1; outdeg[A[v, :]] -= 1; moved = True
        if alive.any():
            score = np.where(alive, outdeg - indeg, -np.inf)
            v = int(np.argmax(score))
            s_head.append(v); alive[v] = False
            indeg[A[:, v]] -= 1; outdeg[A[v, :]] -= 1
    return np.array(s_head + s_tail[::-1])

In [ ]:
def make_C_nilpotente(C: pd.DataFrame):
    """
    Da capire: per ora teniamoci il fatto che rende C nilpotente
    """
    
    C_mat = C.values.astype(int)
    n = C.shape[0]
    order = eades_order(C_mat)
    pos = np.empty(n, int); pos[order] = np.arange(n)
    keep = pos[None, :] < pos[:, None]          # keep[i,j] = (pos[j] < pos[i])
    removed = (C_mat == 1) & (~keep)
    C_nilpotente = C_mat * keep
    C_nilpotente_df = pd.DataFrame(C_nilpotente, index=C.index, columns=C.columns) 
    return C_nilpotente_df

In [1]:
import pandas as pd
import numpy as np

In [2]:
C = pd.read_parquet(str(ROOT / 'data/HS02_all_years/4_digits/alligned/binarized/C/C_4_digits_HS02.parquet'))

In [3]:
A = (C == 1)   # A matrice con valor booleani ---> controllare perchè mi sa che l ho già usata sta cosa
n = A.shape[0]   # numero di righe di C

In [5]:
print (A)
print (n)

hs2002_code_downstream   0101   0102   0103   0104   0105   0106   0201  \
hs2002_code_upstream                                                      
0101                    False  False  False  False  False  False  False   
0102                    False  False  False  False  False  False   True   
0103                    False  False  False  False  False  False  False   
0104                    False  False  False  False  False  False  False   
0105                    False  False  False  False  False   True  False   
...                       ...    ...    ...    ...    ...    ...    ...   
9303                    False  False  False  False  False  False  False   
9304                    False  False  False  False  False  False  False   
9305                    False  False  False  False  False  False  False   
9306                    False  False  False  False  False  False  False   
9307                    False  False  False  False  False  False  False   

hs2002_code_downstream  

In [ ]:
alive = np.ones(n, bool)
indeg  = A.sum(1).astype(float)   # A[i,:] = input di i
outdeg = A.sum(0).astype(float)   # A[:,i] = prodotti di cui i e' input